# PHO Infectious Disease Surveillance Analysis
## 01 | Data Quality Assessment

**Source:** Public Health Ontario (PHO)  
**Study period:** 2016–2025  
**Environment:** Python · Jupyter Notebook · VS Code

### Purpose
Evaluate the structure, completeness, and internal consistency of three related, differently aggregated surveillance exports before further epidemiological analysis.

| Source table | Primary grain | Main use |
|---|---|---|
| Overall | Geography × disease × year | Case counts and reported rates |
| Age & Sex | Geography × disease × year × sex × age group | Demographic stratification |
| Outcome | Ontario × disease × year × outcome type | Deaths and hospitalizations |

**Principles:** Preserve raw inputs; never automatically fill unknown values; keep province-wide totals separate from constituent health units; report unresolved differences instead of silently correcting them.

---
### Step 01 | Environment Setup
Import dependencies and configure compact tabular output.

In [12]:
# ============================================================
# 01 | Environment Setup
# ============================================================
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

print("PHO environment ready.")

PHO environment ready.


### Step 02 | Data Loading & File Paths
**Purpose:** Load all three untouched Excel exports once. The notebook expects to be in `notebooks/` with source files in the sibling `data/` folder.

**Expected output:** A compact three-row loading summary. Raw DataFrames remain available as `df_overall`, `df_age_sex`, and `df_outcome`.

In [13]:
# ============================================================
# 02 | Load Raw Data
# ============================================================
# Resolve the expected project layout without modifying files.
candidates = [Path("../data"), Path("data")]
data_dir = next((p.resolve() for p in candidates if p.is_dir()), None)
if data_dir is None:
    raise FileNotFoundError("Place the Excel files in the project's data/ folder.")

paths = {
    "Overall": data_dir / "Overall.xlsx",
    "Age & Sex": data_dir / "by_sex_age.xlsx",
    "Outcome": data_dir / "outcome.xlsx",
}
for name, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing {name} file: {path}")

df_overall = pd.read_excel(paths["Overall"])
df_age_sex = pd.read_excel(paths["Age & Sex"])
df_outcome = pd.read_excel(paths["Outcome"])
raw_datasets = {"Overall": df_overall, "Age & Sex": df_age_sex, "Outcome": df_outcome}

loading_summary = pd.DataFrame([
    {"Dataset": name, "Rows": len(df), "Columns": df.shape[1]}
    for name, df in raw_datasets.items()
])
display(loading_summary)
print("All three source files loaded; originals unchanged.")

c:\Users\Honar\Downloads\PHO Project\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Dataset,Rows,Columns
0,Overall,20034,7
1,Age & Sex,213741,8
2,Outcome,650,7


All three source files loaded; originals unchanged.


### Step 03 | Data Structure & Initial Inspection
**Purpose:** Verify field names, types, and a small set of representative records.

**Expected output:** One variable dictionary and a two-row preview per dataset. Full tables are deliberately not printed.

In [14]:
# ============================================================
# 03 | Structure & Small Previews
# ============================================================
variable_dictionary = pd.concat([
    pd.DataFrame({
        "Dataset": name, "Column": df.columns,
        "Type": df.dtypes.astype(str).values,
        "Non-null": df.notna().sum().values
    })
    for name, df in raw_datasets.items()
], ignore_index=True)
display(variable_dictionary)

for name, df in raw_datasets.items():
    print(f"{name}: first and last record")
    display(pd.concat([df.head(1), df.tail(1)]))

,Dataset,Column,Type,Non-null
0,Overall,Public health unit,str,20034
1,Overall,Disease,str,20034
2,Overall,Time unit,str,20034
3,Overall,Time period,int64,20034
4,Overall,Cases,int64,20034
5,Overall,Population,int64,20034
6,Overall,"Rate per 100,000",float64,13434
7,Age & Sex,Public health unit,str,213741
8,Age & Sex,Disease,str,213741
9,Age & Sex,Year,int64,213741


Overall: first and last record


,Public health unit,Disease,Time unit,Time period,Cases,Population,"Rate per 100,000"
0,Ontario,Acquired immunodeficiency syndrome (AIDS),Years,2016,66,13876500,0.48
20033,York Region Public Health,Yersiniosis,Years,2025,21,1299177,1.60


Age & Sex: first and last record


,Public health unit,Disease,Year,Sex,Age group,Number of cases,Population,"Rate per 100,000 population"
0,Ontario,Acquired immunodeficiency syndrome (AIDS),2016,Female,0-14,0,"1,097,730.00",0.00
213740,York Region Public Health,Yersiniosis,2025,Male,Unknown,0,NaN,0.00


Outcome: first and last record


,Public health unit,Disease,Outcome,Year,Cases with outcome,Total cases,Percent of cases with outcome
0,Ontario,Acquired immunodeficiency syndrome (AIDS),Deaths,"2,016.00",5.00,66.00,7.58
649,Applied filters:\noperating_name is Ontario\nE...,NaN,NaN,NaN,NaN,NaN,NaN


### Step 04 | Data Quality Assessment
**Purpose:** Screen raw inputs for missing values, exact duplicates, category coverage, unexpected years, and invalid numeric values. These are diagnostic checks, not corrections.

**Expected output:** Two compact tables. Detailed diagnostics are retained in variables for investigation.

In [15]:
# ------------------------------------------------------------
# 04A | Completeness & Duplicate Screening
# ------------------------------------------------------------
missing_report = pd.concat([
    pd.DataFrame({
        "Dataset": name, "Variable": df.columns,
        "Missing": df.isna().sum().values,
        "Missing (%)": (100 * df.isna().mean()).round(2).values
    })
    for name, df in raw_datasets.items()
], ignore_index=True)

quality_overview = pd.DataFrame([
    {
        "Dataset": name, "Rows": len(df),
        "Missing cells": int(df.isna().sum().sum()),
        "Exact duplicates": int(df.duplicated().sum()),
        "Diseases": df["Disease"].nunique(),
        "Geographies": df["Public health unit"].nunique()
    }
    for name, df in raw_datasets.items()
])
print("Quality overview")
display(quality_overview)
print("Fields with missing values")
display(missing_report.loc[missing_report["Missing"] > 0])

Quality overview


,Dataset,Rows,Missing cells,Exact duplicates,Diseases,Geographies
0,Overall,20034,6600,0,67,31
1,Age & Sex,213741,19162,0,67,30
2,Outcome,650,13,0,38,2


Fields with missing values


,Dataset,Variable,Missing,Missing (%)
6,Overall,"Rate per 100,000",6600,32.94
10,Age & Sex,Sex,2749,1.29
13,Age & Sex,Population,16413,7.68
15,Outcome,Public health unit,1,0.15
16,Outcome,Disease,2,0.31
17,Outcome,Outcome,2,0.31
18,Outcome,Year,2,0.31
19,Outcome,Cases with outcome,2,0.31
20,Outcome,Total cases,2,0.31
21,Outcome,Percent of cases with outcome,2,0.31


In [16]:
# ------------------------------------------------------------
# 04B | Temporal & Categorical Coverage
# ------------------------------------------------------------
coverage_summary = []
for name, df in raw_datasets.items():
    year_col = "Time period" if name == "Overall" else "Year"
    years = pd.to_numeric(df[year_col], errors="coerce").dropna()
    coverage_summary.append({
        "Dataset": name,
        "First year": int(years.min()), "Last year": int(years.max()),
        "Years": years.nunique(),
        "Sex categories": df["Sex"].nunique() if "Sex" in df else np.nan,
        "Age categories": df["Age group"].nunique() if "Age group" in df else np.nan,
        "Outcome types": df["Outcome"].nunique() if "Outcome" in df else np.nan,
    })
coverage_report = pd.DataFrame(coverage_summary)
display(coverage_report)

# Data remains unmodified; category labels can be inspected on demand.
age_labels = sorted(df_age_sex["Age group"].dropna().astype(str).unique())
sex_counts = df_age_sex["Sex"].value_counts(dropna=False)
print(f"Age labels: {len(age_labels)} | Sex values (including missing): {len(sex_counts)}")

,Dataset,First year,Last year,Years,Sex categories,Age categories,Outcome types
0,Overall,2016,2025,10,NaN,NaN,NaN
1,Age & Sex,2016,2025,10,3.00,26.00,NaN
2,Outcome,2016,2025,10,NaN,NaN,2.00


Age labels: 26 | Sex values (including missing): 4


### Step 05 | Data Integrity Validation & Controlled Cleaning
**Purpose:** Create audit-friendly working copies, remove verified export footers, align keys, examine demographic reporting, reconcile counts, and validate published numerical indicators.

**Important limitations:** Age categories vary by disease; the notebook explicitly screens for within-group overlapping numeric ranges. Reconciliation results remain provisional until any flagged overlaps or unmatched records are explained. Missing rates and population denominators are not imputed.

**Expected output:** A short cleaning audit, a geography summary, a reconciliation summary, numerical validation results, and a final limitations table.

In [17]:
# ------------------------------------------------------------
# 05A | Working Copies & Export Footer Cleanup
# ------------------------------------------------------------
df_overall_clean = df_overall.copy()
df_age_sex_clean = df_age_sex.copy()
df_outcome_clean = df_outcome.copy()

# Remove only rows confirmed to be non-data (blank/footer).
footer = df_outcome_clean["Public health unit"].astype("string").str.startswith("Applied filters:", na=False)
blank = df_outcome_clean.isna().all(axis=1)
nondata = footer | blank
outcome_excluded = df_outcome_clean.loc[nondata].copy()
df_outcome_clean = df_outcome_clean.loc[~nondata].copy().reset_index(drop=True)

clean_datasets = {"Overall": df_overall_clean, "Age & Sex": df_age_sex_clean, "Outcome": df_outcome_clean}
for name, df in clean_datasets.items():
    cols = [c for c in ("Public health unit", "Disease", "Sex", "Age group", "Outcome", "Time unit") if c in df.columns]
    for col in cols:
        df[col] = df[col].astype("string").str.strip()

# Harmonize year labels on working copies only.
df_overall_clean["Year"] = df_overall_clean["Time period"].astype(int)
df_age_sex_clean["Year"] = df_age_sex_clean["Year"].astype(int)
df_outcome_clean["Year"] = df_outcome_clean["Year"].astype(int)

cleaning_audit = pd.DataFrame([
    {"Dataset": name, "Source rows": len(raw_datasets[name]),
     "Working rows": len(df), "Excluded": len(raw_datasets[name]) - len(df)}
    for name, df in clean_datasets.items()
])
display(cleaning_audit)

,Dataset,Source rows,Working rows,Excluded
0,Overall,20034,20034,0
1,Age & Sex,213741,213741,0
2,Outcome,650,648,2


In [18]:
# ------------------------------------------------------------
# 05B | Geographic & Missing-Value Diagnostics
# ------------------------------------------------------------
regions_overall = set(df_overall_clean["Public health unit"].dropna())
regions_age_sex = set(df_age_sex_clean["Public health unit"].dropna())
regions_outcome = set(df_outcome_clean["Public health unit"].dropna())
shared_regions = regions_overall & regions_age_sex

geography_summary = pd.DataFrame([
    {"Metric": "Overall regions", "Count": len(regions_overall)},
    {"Metric": "Age & Sex regions", "Count": len(regions_age_sex)},
    {"Metric": "Outcome regions", "Count": len(regions_outcome)},
    {"Metric": "Shared Overall / Age & Sex regions", "Count": len(shared_regions)},
])
missing_rate_zero_denominator = int((df_overall_clean["Rate per 100,000"].isna() & df_overall_clean["Population"].eq(0)).sum())
missing_population_positive_cases = int((df_age_sex_clean["Population"].isna() & df_age_sex_clean["Number of cases"].gt(0)).sum())

print("Geographic coverage")
display(geography_summary)
print("Overall-only names:", sorted(regions_overall - regions_age_sex))
print("Missing overall rates with zero denominator:", missing_rate_zero_denominator)
print("Missing age/sex population with positive cases:", missing_population_positive_cases)

Geographic coverage


,Metric,Count
0,Overall regions,31
1,Age & Sex regions,30
2,Outcome regions,1
3,Shared Overall / Age & Sex regions,30


Overall-only names: ['Canada']
Missing overall rates with zero denominator: 6600
Missing age/sex population with positive cases: 2673


In [19]:
# ------------------------------------------------------------
# 05C | Demographic Grain & Age-Overlap Screening
# ------------------------------------------------------------
# A label can overlap another label within a single disease/year/region/sex.
# Exclude labels with unknown or undefined numeric age ranges from this test.
import re

def numeric_interval(label):
    s = str(label).strip()
    if s == "neonatal":
        return (0, 0)  # approximate screening interval, not for harmonization
    match = re.fullmatch(r"(\d+)-(\d+)", s)
    if match:
        return (int(match.group(1)), int(match.group(2)))
    match = re.fullmatch(r"(\d+)\+", s)
    if match:
        return (int(match.group(1)), 200)
    return None

age_mapping = {label: numeric_interval(label) for label in age_labels}
age_key = ["Public health unit", "Disease", "Year", "Sex"]
ages_per_group = df_age_sex_clean.groupby(age_key, dropna=False)["Age group"].unique()
overlap_examples = []
overlap_group_count = 0
for key, labels in ages_per_group.items():
    intervals = [(str(x), age_mapping.get(str(x))) for x in labels]
    intervals = [(label, bounds) for label, bounds in intervals if bounds is not None]
    overlaps = [(a, b) for i,(a,(a0,a1)) in enumerate(intervals)
                for b,(b0,b1) in intervals[i+1:] if max(a0,b0) <= min(a1,b1)]
    if overlaps:
        overlap_group_count += 1
        if len(overlap_examples) < 10:
            overlap_examples.append(dict(zip(age_key, key), Conflicting_labels=overlaps[:3]))

age_overlap_examples = pd.DataFrame(overlap_examples)
demographic_summary = pd.DataFrame([
    {"Metric": "Distinct age labels", "Value": len(age_labels)},
    {"Metric": "Age/sex groups tested", "Value": len(ages_per_group)},
    {"Metric": "Groups with potential overlapping numeric ages", "Value": overlap_group_count},
    {"Metric": "Missing sex records", "Value": int(df_age_sex_clean["Sex"].isna().sum())}
])
display(demographic_summary)
if overlap_group_count:
    display(age_overlap_examples)
print("Note: Different diseases may legitimately use different, non-overlapping age schemes.")

,Metric,Value
0,Distinct age labels,26
1,Age/sex groups tested,26696
2,Groups with potential overlapping numeric ages,0
3,Missing sex records,2749


Note: Different diseases may legitimately use different, non-overlapping age schemes.


In [20]:
# ------------------------------------------------------------
# 05D | Cross-Dataset Case Reconciliation
# ------------------------------------------------------------
comparison_keys = ["Public health unit", "Disease", "Year"]
overall_counts = df_overall_clean.loc[
    df_overall_clean["Public health unit"].isin(shared_regions),
    comparison_keys + ["Cases"]
].rename(columns={"Cases": "Overall Cases"})

age_sex_counts = (df_age_sex_clean.loc[
    df_age_sex_clean["Public health unit"].isin(shared_regions)
].groupby(comparison_keys, as_index=False)["Number of cases"].sum()
 .rename(columns={"Number of cases": "Age & Sex Cases"}))

case_comparison = overall_counts.merge(
    age_sex_counts, on=comparison_keys, how="outer",
    validate="one_to_one", indicator=True
)
case_comparison["Difference"] = case_comparison["Overall Cases"] - case_comparison["Age & Sex Cases"]
case_comparison["Status"] = np.select(
    [case_comparison["_merge"].eq("left_only"),
     case_comparison["_merge"].eq("right_only"),
     case_comparison["Difference"].eq(0)],
    ["Overall Only", "Age & Sex Only", "Match"], default="Mismatch"
)
case_discrepancies = case_comparison.loc[case_comparison["Status"] != "Match"].copy()
case_reconciliation_summary = (case_comparison["Status"].value_counts()
    .rename_axis("Status").reset_index(name="Groups"))

# Zero / positive classification uses whichever table exists, NOT the Overall column alone.
unmatched = case_comparison.loc[case_comparison["Status"].isin(["Overall Only", "Age & Sex Only"])].copy()
unmatched["Available Cases"] = unmatched["Overall Cases"].fillna(unmatched["Age & Sex Cases"])
unmatched_summary = (unmatched.assign(
    Zero=lambda x: x["Available Cases"].eq(0),
    Positive=lambda x: x["Available Cases"].gt(0))
    .groupby("Status").agg(Groups=("Disease", "size"), Zero=("Zero", "sum"), Positive=("Positive", "sum"))
    .reset_index())

print("Overall vs demographic case-count reconciliation")
display(case_reconciliation_summary)
print("Unmatched groups by available case count")
display(unmatched_summary)

Overall vs demographic case-count reconciliation


,Status,Groups
0,Match,13245
1,Overall Only,6104
2,Age & Sex Only,238
3,Mismatch,15


Unmatched groups by available case count


,Status,Groups,Zero,Positive
0,Age & Sex Only,238,0,238
1,Overall Only,6104,6094,10


In [21]:
# ------------------------------------------------------------
# 05E | Numerical Indicator Validation
# ------------------------------------------------------------
valid_denominator = df_overall_clean["Population"].gt(0) & df_overall_clean["Rate per 100,000"].notna()
computed_rate = df_overall_clean.loc[valid_denominator, "Cases"] / df_overall_clean.loc[valid_denominator, "Population"] * 100000
rate_flags = (computed_rate - df_overall_clean.loc[valid_denominator, "Rate per 100,000"]).abs() > 0.05

valid_total = df_outcome_clean["Total cases"].gt(0) & df_outcome_clean["Percent of cases with outcome"].notna()
computed_pct = df_outcome_clean.loc[valid_total, "Cases with outcome"] / df_outcome_clean.loc[valid_total, "Total cases"] * 100
percent_flags = (computed_pct - df_outcome_clean.loc[valid_total, "Percent of cases with outcome"]).abs() > 0.05
outcome_logic_flags = df_outcome_clean["Cases with outcome"].gt(df_outcome_clean["Total cases"])

numeric_validation_summary = pd.DataFrame([
    {"Check": "Overall rates (tolerance: 0.05)", "Checked": int(valid_denominator.sum()), "Flagged": int(rate_flags.sum())},
    {"Check": "Outcome percentages (tolerance: 0.05)", "Checked": int(valid_total.sum()), "Flagged": int(percent_flags.sum())},
    {"Check": "Outcome cases exceed total", "Checked": len(df_outcome_clean), "Flagged": int(outcome_logic_flags.sum())}
])
display(numeric_validation_summary)

,Check,Checked,Flagged
0,Overall rates (tolerance: 0.05),13434,0
1,Outcome percentages (tolerance: 0.05),642,0
2,Outcome cases exceed total,648,0


In [22]:
# ------------------------------------------------------------
# 05F | Final Audit & Review Queue
# ------------------------------------------------------------
mismatches = case_comparison.loc[case_comparison["Status"] == "Mismatch",
    comparison_keys + ["Overall Cases", "Age & Sex Cases", "Difference"]].copy()
positive_unmatched = unmatched.loc[unmatched["Available Cases"].gt(0),
    comparison_keys + ["Status", "Available Cases"]].copy()

final_audit = pd.DataFrame([
    {"Check": "Source files preserved", "Result": "Yes"},
    {"Check": "Outcome non-data rows excluded", "Result": len(outcome_excluded)},
    {"Check": "Exact matches", "Result": int(case_comparison["Status"].eq("Match").sum())},
    {"Check": "Case-count mismatches", "Result": len(mismatches)},
    {"Check": "Unmatched positive-case groups", "Result": len(positive_unmatched)},
    {"Check": "Potential age-overlap groups", "Result": overlap_group_count},
    {"Check": "Rate / outcome percentage flags", "Result": int(rate_flags.sum() + percent_flags.sum())},
])
display(final_audit)
print("Detailed exception tables retained as 'mismatches' and 'positive_unmatched'.")
print("Data readiness: conditional; review outstanding differences and PHO age definitions.")

,Check,Result
0,Source files preserved,Yes
1,Outcome non-data rows excluded,2
2,Exact matches,13245
3,Case-count mismatches,15
4,Unmatched positive-case groups,248
5,Potential age-overlap groups,0
6,Rate / outcome percentage flags,0


Detailed exception tables retained as 'mismatches' and 'positive_unmatched'.
Data readiness: conditional; review outstanding differences and PHO age definitions.


### Findings & Limitations | Review Before EDA

- **Geographic grain:** Overall includes Canada in addition to Ontario and public health units; do not add higher-level totals to component regions.
- **Unmatched records:** Many Overall-only groups may be explicit zero-case combinations absent from the stratified export. Positive unmatched groups require separate review.
- **Demographic definitions:** Age intervals differ by disease. The overlap screen is a diagnostic and does not substitute for PHO data dictionary confirmation.
- **Unavailable denominators:** Keep missing population and undefined rates missing. A zero denominator is not evidence that the true population is zero.
- **Reconciliation:** Even a high match percentage does not establish fully identical coverage. Mpox/CPE count differences remain documented until explained.
- **Reproducibility:** Original exports are never overwritten. Working tables and diagnostic DataFrames are kept in memory.

**Next milestone:** Review `mismatches`, `positive_unmatched`, and flagged age groups; document acceptance rules, then proceed to EDA and dashboard modeling.